<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task2-branch/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import and start pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print("SparkSession started successfully!")

SparkSession started successfully!


In [2]:
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

#READ CSV FILE USING SPARK
df = spark.read.csv('/content/drive/MyDrive/TASK_DATASET/epd_snomed_202603.csv', header=True, inferSchema=True)

print('Dataset Loaded Successfully!')
print(f'no. of record: {df.count():,}')
print(f'no.of columns: {len(df.columns)}')

Mounted at /content/drive
Dataset Loaded Successfully!
no. of record: 18,364,409
no.of columns: 27


In [3]:
#check how many parts the data is split into
print(f'no. of partitions: {df.rdd.getNumPartitions()}')

no. of partitions: 58


In [4]:
#split the partition into 8 section instead
df = df.repartition(8)
print(f'no. of partitions: {df.rdd.getNumPartitions()}')

no. of partitions: 8


In [5]:
#check partition size
from pyspark.sql.functions import spark_partition_id, count

df.groupBy(spark_partition_id().alias("partition_id")) \
  .count() \
  .orderBy("partition_id") \
  .show()

+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|2295550|
|           1|2295550|
|           2|2295554|
|           3|2295553|
|           4|2295552|
|           5|2295550|
|           6|2295552|
|           7|2295548|
+------------+-------+



In [6]:
#cleaning the dataset

#drop columns that are not required for prediction of cost of prescription
drop_columns = ['ADDRESS_1', 'ADDRESS_2', 'ADDRESS_3', 'ADDRESS_4',
    'UNIDENTIFIED', 'YEAR_MONTH',
    'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_PRESENTATION_CODE',
    'PRACTICE_CODE', 'PCO_CODE', 'ICB_CODE',
    'REGIONAL_OFFICE_CODE']

df = df.drop(*drop_columns)
print('columns after dropping - ')
print(df.columns)

columns after dropping - 
['REGIONAL_OFFICE_NAME', 'ICB_NAME', 'PCO_NAME', 'PRACTICE_NAME', 'POSTCODE', 'BNF_CHEMICAL_SUBSTANCE', 'BNF_PRESENTATION_NAME', 'BNF_CHAPTER_PLUS_CODE', 'QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'SNOMED_CODE']


In [7]:
print(f'no.of columns: {len(df.columns)}')

no.of columns: 15


In [8]:
#fix empty value
from pyspark.sql.functions import col, isnan, when, sum as spark_sum

#check missing values
print('Missing values as per columns- ')
df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

#drop rows with missing ACTUAL_COST value - because a row with no cost is useless for training
df = df.dropna(subset=['ACTUAL_COST'])

# Fill missing POSTCODE
df = df.fillna({'POSTCODE': 'UNKNOWN'})

# fill remaining null values with 0
df = df.fillna({
    'QUANTITY': 0.0,
    'ITEMS': 0,
    'TOTAL_QUANTITY': 0.0,
    'ADQ_USAGE': 0.0,
    'NIC': 0.0,
    'SNOMED_CODE': 0
})

# Verify no missing values are remaining
print("Missing values AFTER cleaning:")
df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

#print rows left after cleaning
print(f"No. of Records after cleaning: {df.count():,}")

Missing values as per columns- 
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+---+-----------+-----------+
|REGIONAL_OFFICE_NAME|ICB_NAME|PCO_NAME|PRACTICE_NAME|POSTCODE|BNF_CHEMICAL_SUBSTANCE|BNF_PRESENTATION_NAME|BNF_CHAPTER_PLUS_CODE|QUANTITY|ITEMS|TOTAL_QUANTITY|ADQ_USAGE|NIC|ACTUAL_COST|SNOMED_CODE|
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+---+-----------+-----------+
|                   0|       0|       0|            0|    6793|                     0|                    0|                    0|       0|    0|             0|        0|  0|          0|      11449|
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+--

In [9]:
#Encoding
#import stringindexer or converting categorical text columns into numeric indices
from pyspark.ml.feature import StringIndexer

#columns to be encoded for model training
columns_to_encoded = ['REGIONAL_OFFICE_NAME',
    'ICB_NAME',
    'PCO_NAME',
    'PRACTICE_NAME',
    'BNF_CHEMICAL_SUBSTANCE',
    'BNF_PRESENTATION_NAME',
    'BNF_CHAPTER_PLUS_CODE']

#create string index for each column
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_IDX",
        handleInvalid="keep" #unseen or null values are assigned a separate index
    )
    for col in columns_to_encoded
]

print(f"{len(indexers)} columns encoded!")

7 columns encoded!


In [12]:
#vector assembler
from pyspark.ml.feature import VectorAssembler
#combine all feature columns for ML - fit all columns used for prediction into one
features = [ 'QUANTITY', # How many units were prescribed
           'ITEMS', # Number of prescription items
            'TOTAL_QUANTITY', # Total quantity across all items
            'ADQ_USAGE', # Average Daily Quantity usage
            'NIC', # Net Ingredient Cost (closely related to actual cost)
            'SNOMED_CODE', # Clinical code for the drug
            'REGIONAL_OFFICE_NAME_IDX', # Region encoded as number
            'ICB_NAME_IDX', # Integrated Care Board encoded as number
            'PCO_NAME_IDX', # Primary Care Organisation encoded as number
            'PRACTICE_NAME_IDX', # GP Practice encoded as number
            'BNF_CHEMICAL_SUBSTANCE_IDX', # Drug name encoded as number
            'BNF_PRESENTATION_NAME_IDX', # Drug form encoded as number
            'BNF_CHAPTER_PLUS_CODE_IDX' # Drug category encoded as number
            ]

#using assembler to combine all features into one vector
assembler = VectorAssembler(inputCols= features,
                            outputCol="features", # Output column name
                            handleInvalid="keep" # to keep rows with invalid values instead of crashing
                             )

print("VectorAssembler created successfully!")

VectorAssembler created successfully!


In [14]:
#scaling
#StandardScaler rescales everything to the same range for fair comparison
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",           # Takes features vector from previous cell
    outputCol="scaled_features",   # gives rescaled version
    withStd=True,                  # with std means divide by standard deviation — normalises spread
    withMean=False                 # Don't subtract mean — keeps sparse data efficient
)

print("StandardScaler created successfully!")

StandardScaler created successfully!
